# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

GOOGLE_API_KEY  = os.getenv('GOOGLE_API_KEY') 

### Example 1

#### Create our QandA application

In [2]:
# 1. Full clean uninstall (includes corrupted packages)
!pip uninstall langchain langchain-core langchain-community langchain-google-genai langchain-huggingface langchain-text-splitters langchain-text-splitters-* -y

# 2. Clear pip cache
!pip cache purge

# 3. Install compatible pinned versions
!pip install "langchain-core>=0.2.28,<0.3" langchain-google-genai langchain-huggingface "langchain-community>=0.2.16,<0.3" langchain-text-splitters

# 4. Verify
!pip list | grep langchain

zsh:1: no matches found: langchain-text-splitters-*
Files removed: 271 (31.0 MB)
Directories removed: 5
langchain                                0.2.17
langchain-classic                        1.0.1
langchain-community                      0.2.19
langchain-core                           0.2.43
langchain-elasticsearch                  1.0.0
langchain-experimental                   0.4.1
langchain-google-genai                   1.0.10
langchain-huggingface                    0.0.3
langchain-openai                         1.1.11
langchain-pinecone                       0.2.13
langchain-text-splitters                 0.2.4


In [3]:
import sys
!{sys.executable} -m pip list | findstr langchain

zsh:1: command not found: findstr
ERROR: Pipe to stdout was broken


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain  # ✅ FIXED
from langchain_core.output_parsers import StrOutputParser, BaseOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langchain.evaluation.qa import QAGenerateChain

In [5]:
file = r'/Users/ia_dev/Desktop/Lab Control/lab-langchain-evaluation/data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(data)

# Create embeddings and vectorstore
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

vectorstore = InMemoryVectorStore.from_documents(
    docs, embeddings  # embeds and indexes documents
)

retriever = vectorstore.as_retriever()

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

<<<<>>>>>

QUESTION: {question}

ANSWER:
""")

qa = create_stuff_documents_chain(llm, prompt, document_separator="<<<<>>>>>")

# Fixed chain - provides both context AND question
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa
)


result = chain.invoke("Do the Cozy Comfort Pullover Set have side pockets?")
print(result)

Yes, the Cozy Comfort Pullover Set has side pockets on the pull-on pants.


#### Coming up with test datapoints

In [21]:
data[10]

Document(metadata={'source': '/Users/ia_dev/Desktop/Lab Control/lab-langchain-evaluation/data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [9]:
data[11]

Document(metadata={'source': '/Users/ia_dev/Desktop/Lab Control/lab-langchain-evaluation/data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [10]:
!pip install langchain_classic

I0000 00:00:1773311873.091381  310153 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core━━━━━━━━━━ 0/3 [langsmith]
    Found existing installation: langchain-core 0.2.430/3 [langsmith]
    Uninstalling langchain-core-0.2.43:━━━━━ 0/3 [langsmith]
      Successfully uninstalled langchain-core-0.2.432m0/3 [langsmith]
  Attempting uninstall: langchain-text-splitters━━━━━━━━━━━━━━━━━━ 1/3 [langchain-core]
    Found existing installation: langchain-text-splitters 0.2.4 1/3 [langchain-core]
    Uninstalling langchain-text-splitters-0.2.4:━━━━━━━━━━━━━━ 1/3 [langchain-core]
      Successfully uninstalled langchain-text-splitters-0.2.4━ 1/3 [langchain-core]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain-text-splitters]ore]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the f

In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)


class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")  
llm_chain = prompt_template | llm | AnswerOutputParser() 

query = "Is the Cozy Comfort Pullover Set available in different colors?"
result = llm_chain.invoke({"query": query})  
print(result.answer)  

Yes


In [23]:
# Run this to see what's in langchain.evaluation
import langchain.evaluation.qa
print(dir(langchain.evaluation.qa))

['ContextQAEvalChain', 'CotQAEvalChain', 'QAEvalChain', 'QAGenerateChain', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'eval_chain', 'eval_prompt', 'generate_chain', 'generate_prompt']


#### LLM-Generated examples

In [24]:
from langchain.evaluation.qa import QAGenerateChain  
from langchain.chains import LLMChain  

example_gen_chain = QAGenerateChain.from_llm(ChatGoogleGenerativeAI(model="gemini-2.5-flash"))

# Your EXACT workflow now works!
llm_chain = LLMChain(llm=llm, prompt=prompt_template, output_parser=AnswerOutputParser())

new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

d_flattened = [data['qa_pairs'] for data in new_examples]


/opt/anaconda3/lib/python3.13/site-packages/langchain/chains/llm.py:366: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


#### Combine examples

In [25]:
# examples += new_example
examples += d_flattened

In [26]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What material is used for the innersole of the Women's Campside Oxfords, and what specific technology is incorporated into it for odor control?",
  'answer': "The innersole of the Women's Campside Oxfords is made from comfortable EVA and incorporates Cleansport NXT® antimicrobial odor control."},
 {'query': 'What materials are used in the construction of the Recycled Waterhog Dog Mat, and what percentage of those materials are recycled?',
  'answer': 'The Recycled Waterhog Dog Mat is constructed from 24 oz. polyester fabric, 94% of which is made from recycled materials, and it also features a rubber backing.'},
 {'query': "What are the key features and benefits of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece, specifically regarding its fabric, sun 

In [27]:
result = qa.invoke({
    "question": examples[6]["query"],  # ← "question" not "input"
    "context": retriever.invoke(examples[6]["query"])
})
print(result)

The new technology featured in the EcoFlex 3L Storm Pants is **TEK O2 technology**.

It enhances the pants' performance by making them even more breathable, offering the most breathability tested. This technology is guaranteed to keep the user dry and comfortable, whatever the activity and whatever the weather, by being air-permeable, allowing air in and keeping water out.


### Manual Evaluation - Fun part

In [28]:
import langchain
langchain.debug = True

# qa needs BOTH question AND context
result = qa.invoke({
    "question": examples[0]["query"],    
    "context": retriever.invoke(examples[0]["query"])  
})
print(result)

[chain/start] [chain:stuff_documents_chain] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context>] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] Entering Chain run with input:
[inputs]
[chain/end] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] s] Exiting Chain run with output:
{
  "output": "On-seam pockets, with a zippered security pocket on the right side.\n\nAll pockets have sturdy pocket bags and offer plenty of room for a wallet, cell phone and more.\n\nGusseted crotch for ease of movement.\n\nImported.<<<<>>>>>: 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name.

In [29]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [30]:
# predictions must be list of dicts like:
# [{'query': '...', 'answer': '...', 'result': '...'}, ...]

predictions = []
for i, eg in enumerate(examples[:5]):  # Limit for testing
    docs = retriever.invoke(eg["query"])
    result = qa.invoke({"question": eg["query"], "context": docs})
    
    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result
    }
    predictions.append(pred_dict)

# Now your print loop works perfectly!
for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set has side pockets on the pull-on pants.

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 2:
Question: What material is used for the innersole of the Women's Campside Oxfords, and what specific technology is incorporated into it for odor control?
Real Answer: The innersole of the Women's Campside Oxfords is made from comfortable EVA and incorporates Cleansport NXT® antimicrobial odor control.
Predicted Answer: The innersole of the Women's Campside Oxfords is made of comfortable EVA, and it incorporates Cleansport NXT® antimicrobial odor control technology.

Example 3:
Question: What materials are used in the construction of the Recycled Waterhog Dog Mat, and what 

In [31]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Simple eval prompt (no classes)
eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (0-1, higher=better match).

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()  # Simple string parser

# Your EXACT workflow (no classes)
graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    score = float(score_raw.strip())  # Extract number
    
    graded_outputs.append({'score': score})
    
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

ValueError: could not convert string to float: 'Score: 1.0'

In [ ]:
graded_outputs

[{'score': 1.0},
 {'score': 1.0},
 {'score': 1.0},
 {'score': 0.9},
 {'score': 1.0}]

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_core.runnables import RunnablePassthrough

# Load the text file
loader = TextLoader(r"C:\Users\Denish\Desktop\TA\Week_8_Lab_Soluiton\Day_2\nyc_text.txt")
docs = loader.load()

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)

# Create vectorstore
vectorstore = InMemoryVectorStore.from_documents(
    split_docs, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

# Create QA chain
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

QUESTION: {question}

ANSWER:
""")

qa_chain = create_stuff_documents_chain(llm, prompt)

# Combine into runnable chain that returns both answer and source documents
qa_chain_with_docs = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa_chain
)



In [ ]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain_with_docs.invoke(question)
print(result)

New York City was named after King Charles II of England granted the lands to his brother, the Duke of York.


Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [ ]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [ ]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [ ]:

from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

all_predictions = []
for example in examples:
    q = example["query"]
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})
    all_predictions.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": example["ground_truths"][0]
    })

dataset_all = Dataset.from_list(all_predictions)

print("evaluating faithfulness...")
f = evaluate(dataset_all, metrics=[faithfulness])
print(f)

evaluating faithfulness...


Evaluating: 100%|██████████| 5/5 [00:06<00:00,  1.25s/it]


{'faithfulness': 0.8000}


1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

**Faithfulness**

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [ ]:
print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

evaluating context recall...


Evaluating: 100%|██████████| 5/5 [00:03<00:00,  1.26it/s]


{'context_recall': 0.5733}


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness
from datasets import Dataset

# Use question 2 (economic significance) — produces long detailed answer
high_data = Dataset.from_list([all_predictions[2]])  # economic significance question
high_score = evaluate(high_data, [faithfulness])["faithfulness"]
print("HIGH Faithfulness (real answer):", high_score)

# LOW — contradicting claims
fake = all_predictions[2].copy()
fake["answer"] = "New York City has no economic importance. It is a small rural town with no financial institutions, no corporations, and no international trade. Wall Street closed in 1990."
low_data = Dataset.from_list([fake])
low_score = evaluate(low_data, [faithfulness])["faithfulness"]
print("LOW Faithfulness (fake answer):", low_score)

print("\n--- Explanation ---")
print(f"Real answer score:  {high_score:.2f}  ← claims supported by context")
print(f"Fake answer score:  {low_score:.2f}  ← claims contradict context")

Evaluating: 100%|██████████| 1/1 [00:05<00:00,  5.37s/it]


HIGH Faithfulness (real answer): 1.0


Evaluating: 100%|██████████| 1/1 [00:04<00:00,  4.78s/it]


LOW Faithfulness (fake answer): 0.0

--- Explanation ---
Real answer score:  1.00  ← claims supported by context
Fake answer score:  0.00  ← claims contradict context


In [ ]:
print("Question:", all_predictions[2]["question"])
print("Answer:", all_predictions[2]["answer"])

Question: What is the economic significance of New York City?
Answer: New York City is a global hub of business and commerce, with a diverse economy that includes banking and finance, health care, technology, retail, tourism, real estate, media, advertising, legal services, and more. It is a major economic engine with a gross metropolitan product of over $2.4 trillion, making it the largest metropolitan economy in the world. Additionally, New York City is home to the highest number of billionaires, individuals of ultra-high net worth, and millionaires of any city in the world, making it an established safe haven for global investors.


**Context Relevancy**

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [ ]:
from ragas.metrics import context_recall

# HIGH — real docs
high_data = Dataset.from_list([all_predictions[2]])
high_score = evaluate(high_data, [context_recall])["context_recall"]
print("HIGH Context Recall (real docs):", high_score)

# LOW — off-topic docs
fake = all_predictions[2].copy()
fake["contexts"] = [
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language used in data science."
]
low_data = Dataset.from_list([fake])
low_score = evaluate(low_data, [context_recall])["context_recall"]
print("LOW Context Recall (fake docs):", low_score)

print("\n--- Explanation ---")
print(f"Real docs score:  {high_score:.2f}  ← ground truth found in context")
print(f"Fake docs score:  {low_score:.2f}  ← ground truth absent from context")

Evaluating: 100%|██████████| 1/1 [00:03<00:00,  3.61s/it]


HIGH Context Recall (real docs): 0.2


Evaluating: 100%|██████████| 1/1 [00:02<00:00,  2.55s/it]


LOW Context Recall (fake docs): 0.0

--- Explanation ---
Real docs score:  0.20  ← ground truth found in context
Fake docs score:  0.00  ← ground truth absent from context


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

# helper
def get_contexts_str(contexts):
    return [
        doc.page_content if hasattr(doc, "page_content") else str(doc)
        for doc in contexts
    ] if contexts and hasattr(contexts[0], "page_content") else contexts

def evaluate_answer(question, answer, contexts, ground_truth=None, test_name=""):
    contexts_str = get_contexts_str(contexts)

    data = {
        "question": [question],
        "answer":   [answer],
        "contexts": [contexts_str],
        "ground_truth": [ground_truth if ground_truth else answer]
    }

    dataset = Dataset.from_dict(data)
    results = evaluate(dataset=dataset, metrics=[faithfulness, answer_relevancy, context_recall])

    print(f"\n{'='*60}")
    print(f"TEST: {test_name}")
    print(f"Faithfulness:     {results['faithfulness']:.4f}")
    print(f"Answer Relevancy: {results['answer_relevancy']:.4f}")
    print(f"Context Recall:   {results['context_recall']:.4f}")
    return results

# Test it
evaluate_answer(
    question=all_predictions[2]["question"],
    answer=all_predictions[2]["answer"],
    contexts=all_predictions[2]["contexts"],
    ground_truth=all_predictions[2]["ground_truth"],
    test_name="Economic Significance"
)

Evaluating: 100%|██████████| 3/3 [00:08<00:00,  2.84s/it]



TEST: Economic Significance
Faithfulness:     1.0000
Answer Relevancy: 0.9215
Context Recall:   0.2000


{'faithfulness': 1.0000, 'answer_relevancy': 0.9215, 'context_recall': 0.2000}

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [ ]:
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

dataset_all = Dataset.from_list(all_predictions)

results = evaluate(
    dataset_all,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

print(results)
results.to_pandas()

Evaluating: 100%|██████████| 20/20 [00:08<00:00,  2.30it/s]


{'faithfulness': 0.7000, 'answer_relevancy': 0.9381, 'context_precision': 0.9667, 'context_recall': 0.5733}


,question,answer,contexts,ground_truth,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the population of New York City as of ...,The population of New York City as of 2020 is ...,[New York City is the most populous city in th...,"8,804,190",1.0,1.000000,1.000000,1.000000
1,Which borough of New York City has the highest...,Manhattan (New York County),[New York City is the most populous city in th...,Brooklyn,0.0,0.911307,0.916667,0.000000
2,What is the economic significance of New York ...,New York City is a global hub of business and ...,[Despite New York's heavy reliance on its vast...,"New York City's economic significance is vast,...",1.0,0.921540,0.916667,0.200000
3,How did New York City get its name?,New York City was named after King Charles II ...,[The city and its metropolitan area constitute...,New York City got its name when it came under ...,0.5,0.918478,1.000000,1.000000
4,What is the significance of the Statue of Libe...,The Statue of Liberty in New York City is a sy...,[the Dutch in July 1673 and was renamed New Or...,The Statue of Liberty in New York City holds g...,1.0,0.939276,1.000000,0.666667


In [ ]:
# run the queries as a batch for efficiency
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

# Step 1 — batch predictions (same idea as qa_chain.batch(examples))
all_preds = []
for example in examples:
    q = example["query"]
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})
    all_preds.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": example["ground_truths"][0]
    })

dataset_all = Dataset.from_list(all_preds)

# Step 2 — replaces faithfulness_chain.evaluate(examples, predictions)
print("evaluating faithfulness...")
r = evaluate(dataset_all, metrics=[faithfulness])
print(r)

# Step 3 — replaces context_recall_chain.evaluate(examples, predictions)
print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

evaluating faithfulness...


Evaluating: 100%|██████████| 5/5 [00:06<00:00,  1.24s/it]


{'faithfulness': 0.8000}
evaluating context recall...


Evaluating: 100%|██████████| 5/5 [00:04<00:00,  1.23it/s]


{'context_recall': 0.5733}
